# commit-model — QLoRA fine-tuning on Kaggle

Fine-tunes `Qwen2.5-Coder-3B-Instruct` with DoRA (4-bit QLoRA) to generate
Conventional Commit messages from git diffs.

**GPU**: T4 x1 or P100 (16 GB VRAM) — enable in notebook settings before running.

**Internet**: must be ON to download the base model from HuggingFace Hub.

After training completes, download the adapter from `/kaggle/working/adapters/`
via the Output tab.

In [ ]:
# Install dependencies (Kaggle has torch pre-installed; we pin the rest)
!pip install transformers>=4.47.0 peft>=0.12.0 bitsandbytes>=0.43.0 \
             accelerate>=0.33.0 datasets>=2.20.0 safetensors pyyaml -q

In [ ]:
import subprocess, sys, os
from pathlib import Path

# Install the commit_model package from this repo
repo_root = Path("/kaggle/working/commit-model-kaggle")
if not repo_root.exists():
    # When the kernel is pushed via API the repo files land in /kaggle/working/
    repo_root = Path("/kaggle/working")

subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(repo_root), "-q"], check=True)
print("commit_model package installed")

In [ ]:
# Point paths at Kaggle locations
# Replace YOUR_KAGGLE_USERNAME with your actual Kaggle username.
DATASET_NAME = "YOUR_KAGGLE_USERNAME/commit-model-data"
DATA_DIR = f"/kaggle/input/{DATASET_NAME.split('/')[1]}"
ADAPTER_DIR = "/kaggle/working/adapters"

import yaml
config_path = repo_root / "configs" / "lora_config_kaggle.yaml"
with open(config_path) as f:
    config = yaml.safe_load(f)

config["data"] = DATA_DIR
config["adapter_path"] = ADAPTER_DIR

# Write the patched config so train_lora.py picks it up
patched_config_path = "/kaggle/working/lora_config_kaggle_runtime.yaml"
with open(patched_config_path, "w") as f:
    yaml.dump(config, f)

print(f"Data:    {DATA_DIR}")
print(f"Adapter: {ADAPTER_DIR}")

In [ ]:
# Verify CUDA and data are accessible before starting the long run
import torch
import os

assert torch.cuda.is_available(), "GPU not available — enable GPU in notebook settings"
gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU: {gpu} ({vram:.1f} GB VRAM)")

for split in ["train.jsonl", "valid.jsonl"]:
    p = os.path.join(DATA_DIR, split)
    assert os.path.exists(p), f"Missing: {p}  — did the dataset upload succeed?"
    with open(p) as fh:
        n = sum(1 for _ in fh)
    print(f"{split}: {n:,} examples")

In [ ]:
# Run training
train_script = str(repo_root / "scripts" / "train_lora.py")
subprocess.run(
    [sys.executable, train_script, "--config", patched_config_path],
    check=True,
)

In [ ]:
# Quick smoke test after training
sample_script = str(repo_root / "scripts" / "generate_sample.py")
subprocess.run(
    [sys.executable, sample_script,
     "--adapter-path", ADAPTER_DIR],
    check=True,
)

## Download the trained adapter

After the training cell completes:
1. Go to the **Output** tab on the right side of this notebook.
2. Navigate to `adapters/` and download the files.
3. Or use the Kaggle API locally:
   ```bash
   kaggle kernels output YOUR_KAGGLE_USERNAME/commit-model-training -p ./models/adapters/
   ```